# FATLAS: Visceral Fat Research Multi-Agent Notebook

**FATLAS** = **Fat Tissue Analysis & Learning Agent System**

This Google Colab notebook sets up a small AI research team:

- **Research Coordinator Agent** — receives all findings, compares them, identifies cross-links, and proposes next research directions.
- **Hormone Signaling Agent**
- **Genetics & Epigenetics Agent**
- **Gut Microbiome Agent**
- **Inflammation & Immune Agent**
- **Aging & Metabolic Change Agent**

## Scientific mission

**Research question:**  
Why does the body preferentially store fat around the abdomen in some people but not others?

**Goal:**  
Help scientists better understand signaling pathways that tell abdominal fat cells, especially visceral adipocytes, to store or retain energy, then identify safe ways to reduce that signal.

## Safety note

This notebook is for research organization and hypothesis generation. It is **not medical advice**, does not diagnose or treat disease, and should not be used to recommend unsafe weight-loss methods.

## 1. Install and import libraries

In Colab, run this cell first.

In [ ]:
# Colab-safe install cell
# Do NOT use -U here. Upgrading everything can break Colab's pinned packages.
# This keeps pandas and google-auth aligned with Colab, while keeping google-genai below 2.x.

!pip install -q "pandas==2.2.2" "google-auth==2.47.0"
!pip install -q --no-deps "google-genai>=1.64.0,<2.0.0"

print("Install complete. If you previously installed incompatible versions, restart the runtime, then run all cells again.")


### If you already got dependency conflict errors

After running the install cell above, go to:

**Runtime → Restart runtime**

Then run the notebook again from the top.

This matters because Colab may already have loaded the incompatible package versions into memory.


In [ ]:
import os
import json
from datetime import datetime
from pathlib import Path

import pandas as pd
from google import genai

try:
    from google.colab import userdata
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("Setup complete.")
print("Running in Colab:", IN_COLAB)

## 2. Add your Gemini API key

In Google Colab:

1. Click the **key icon** on the left sidebar, called **Secrets**.
2. Add a secret named: `GOOGLE_API_KEY`
3. Paste your Gemini API key from Google AI Studio.
4. Turn on notebook access for the secret.

Then run the cell below.

In [ ]:
def get_google_api_key():
    if IN_COLAB:
        key = userdata.get("GOOGLE_API_KEY")
    else:
        key = os.environ.get("GOOGLE_API_KEY")

    if not key:
        raise ValueError(
            "No GOOGLE_API_KEY found. In Colab, add it under Secrets as GOOGLE_API_KEY."
        )

    # Remove accidental spaces, tabs, and line breaks copied into Colab Secrets.
    key = "".join(str(key).split())

    # Gemini API keys from Google AI Studio usually start with AIza.
    if not key.startswith("AIza"):
        raise ValueError(
            "The value stored in GOOGLE_API_KEY does not look like a Google AI Studio Gemini API key. "
            "It should usually start with 'AIza'. Go to https://aistudio.google.com/apikey, create/copy an API key, "
            "and replace the current Colab Secret value."
        )

    return key

GOOGLE_API_KEY = get_google_api_key()
client = genai.Client(api_key=GOOGLE_API_KEY)

print("Gemini client ready.")
print("Key format check passed:", GOOGLE_API_KEY[:6] + "..." + GOOGLE_API_KEY[-4:])

### API key troubleshooting

If you see `Illegal header value`, the key stored in Colab Secrets probably has a hidden line break, space, or the wrong kind of token.

A valid Gemini API key from Google AI Studio usually starts with:

`AIza`

If your secret starts with something like `AQ...`, it is probably not the right API key. Replace it with a key from:

https://aistudio.google.com/apikey

## 3. Project configuration

You can change the model if needed. Start with the default below.

In [ ]:
MODEL_NAME = "gemini-2.5-flash"

PROJECT_TITLE = "FATLAS Visceral Fat Signaling Pathway Study"

RESEARCH_TOPIC = '''
Why does the body preferentially store fat around the abdomen in some people but not others?
'''

PROJECT_GOAL = '''
Help scientists better understand the signaling pathways that tell fat cells in the abdomen,
especially visceral adipocytes, to store or hold onto energy, then devise safe ways to reduce
or switch off that signal without causing harm.
'''

OUTPUT_DIR = Path("fatlas_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print(PROJECT_TITLE)

## 4. Helper functions

These functions call Gemini, clean JSON output, and save reports.

In [ ]:
def call_gemini(prompt, model=MODEL_NAME, temperature=0.2):
    """Call Gemini and return text."""
    response = client.models.generate_content(
        model=model,
        contents=prompt,
        config={
            "temperature": temperature,
        },
    )
    return response.text


def extract_json(text):
    """Try to extract JSON even if the model wraps it in markdown."""
    cleaned = text.strip()

    if cleaned.startswith("```json"):
        cleaned = cleaned.replace("```json", "", 1).strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.replace("```", "", 1).strip()
    if cleaned.endswith("```"):
        cleaned = cleaned[:-3].strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        start = cleaned.find("{")
        end = cleaned.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(cleaned[start:end+1])
        raise


def save_json(data, filename):
    path = OUTPUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)
    return path


def save_markdown(text, filename):
    path = OUTPUT_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

## 5. Define the specialist AI agents

Each specialist gets the same mission, but a different focus area.

In [ ]:
AGENTS = {
    "Hormone Signaling Agent": {
        "focus": "Insulin, cortisol, leptin, adiponectin, sex hormones, thyroid signaling, hunger/satiety signaling, and metabolic regulation.",
        "questions": [
            "How does insulin resistance promote visceral fat accumulation?",
            "How does chronic cortisol signaling influence abdominal fat?",
            "What roles do leptin resistance and adiponectin play?",
            "How do menopause, testosterone decline, thyroid changes, and chronic stress affect fat distribution?"
        ]
    },
    "Genetics & Epigenetics Agent": {
        "focus": "Inherited risk, gene expression, fat depot differences, adipocyte programming, epigenetic changes, and gene-environment interactions.",
        "questions": [
            "Which genes are associated with visceral adiposity?",
            "How do visceral fat cells differ from subcutaneous fat cells at the gene-expression level?",
            "Can abdominal fat-cell behavior be safely reprogrammed?",
            "Which genetic pathways overlap with insulin resistance, inflammation, and aging?"
        ]
    },
    "Gut Microbiome Agent": {
        "focus": "Gut bacteria, short-chain fatty acids, bile acids, gut barrier integrity, endotoxins, microbial metabolites, and energy harvest.",
        "questions": [
            "Which microbiome patterns correlate with visceral fat?",
            "Does gut permeability or endotoxin exposure trigger abdominal fat inflammation?",
            "How do microbial metabolites affect fat storage and insulin sensitivity?",
            "Could diet, prebiotics, probiotics, or microbiome therapies safely reduce visceral fat signaling?"
        ]
    },
    "Inflammation & Immune Agent": {
        "focus": "Macrophages, cytokines, inflammasomes, adipose tissue inflammation, fibrosis, immune cell infiltration, and chronic low-grade inflammation.",
        "questions": [
            "How does visceral fat become inflamed?",
            "Which inflammatory signals tell fat cells to hold energy?",
            "How do macrophages and cytokines affect adipocyte metabolism?",
            "Can inflammation be reduced without dangerous immune suppression?"
        ]
    },
    "Aging & Metabolic Change Agent": {
        "focus": "Age-related fat redistribution, sarcopenia, mitochondrial dysfunction, brown/beige fat decline, senescent cells, hormonal aging, and reduced metabolic flexibility.",
        "questions": [
            "Why does visceral fat tend to increase with age?",
            "How do declining sex hormones, muscle loss, and cellular senescence contribute?",
            "How does aging affect mitochondrial function and thermogenesis?",
            "Can younger fat-cell behavior be restored safely?"
        ]
    }
}

list(AGENTS.keys())

## 6. Specialist agent prompt template

The agents are instructed to avoid overclaiming and separate evidence from hypothesis.

In [ ]:
SPECIALIST_PROMPT_TEMPLATE = """
You are the {agent_name} in the FATLAS visceral fat research project.

Project topic:
{research_topic}

Project goal:
{project_goal}

Your focus area:
{focus}

Your assigned questions:
{questions}

Instructions:
- Use established biomedical reasoning.
- Separate well-supported findings from hypotheses.
- Do not invent citations. If you are unsure of a source, say source verification needed.
- Do not provide personal medical advice.
- Do not recommend unsafe weight-loss methods.
- Focus on mechanisms, pathways, biomarkers, unknowns, and possible safe intervention targets.
- Make your output useful for a Research Coordinator Agent that will compare all reports.

Return ONLY valid JSON in this schema:

{{
  "agent_name": "{agent_name}",
  "focus_area": "{focus}",
  "executive_summary": "",
  "key_mechanisms": [
    {{
      "mechanism": "",
      "pathway_or_molecules": [],
      "how_it_may_affect_visceral_fat": "",
      "evidence_strength": "high | moderate | emerging | speculative",
      "safety_notes": ""
    }}
  ],
  "cross_links_to_other_domains": [
    {{
      "linked_domain": "hormones | genetics | microbiome | inflammation | aging | other",
      "connection": ""
    }}
  ],
  "candidate_biomarkers": [],
  "possible_safe_intervention_targets": [
    {{
      "target": "",
      "rationale": "",
      "risk_or_caution": ""
    }}
  ],
  "major_unknowns": [],
  "suggested_experiments": [
    {{
      "experiment": "",
      "what_it_tests": "",
      "ideal_data": ""
    }}
  ],
  "source_search_terms_for_human_review": []
}}
"""

## 7. Run all specialist agents

This will call Gemini once per agent and save each report as JSON.

In [ ]:
def run_specialist_agent(agent_name, agent_info):
    prompt = SPECIALIST_PROMPT_TEMPLATE.format(
        agent_name=agent_name,
        research_topic=RESEARCH_TOPIC,
        project_goal=PROJECT_GOAL,
        focus=agent_info["focus"],
        questions=json.dumps(agent_info["questions"], indent=2)
    )

    raw = call_gemini(prompt)
    try:
        data = extract_json(raw)
    except Exception as e:
        print(f"JSON parsing failed for {agent_name}. Saving raw output.")
        data = {
            "agent_name": agent_name,
            "parse_error": str(e),
            "raw_output": raw
        }

    safe_name = agent_name.lower().replace(" ", "_").replace("&", "and")
    save_json(data, f"{safe_name}.json")
    return data


specialist_reports = []

for agent_name, agent_info in AGENTS.items():
    print(f"Running: {agent_name}")
    report = run_specialist_agent(agent_name, agent_info)
    specialist_reports.append(report)

print(f"Completed {len(specialist_reports)} specialist reports.")

## 8. Review specialist summaries

This creates a simple table so you can quickly inspect the agent findings.

In [ ]:
summary_rows = []

for report in specialist_reports:
    summary_rows.append({
        "Agent": report.get("agent_name", ""),
        "Focus": report.get("focus_area", "")[:120],
        "Executive Summary": report.get("executive_summary", "")[:300],
        "Mechanisms Count": len(report.get("key_mechanisms", [])),
        "Unknowns Count": len(report.get("major_unknowns", [])),
        "Experiments Count": len(report.get("suggested_experiments", [])),
    })

df_summaries = pd.DataFrame(summary_rows)
df_summaries

## 9. Research Coordinator Agent

The Coordinator receives all specialist reports and looks for meaning across the data.

In [ ]:
COORDINATOR_PROMPT_TEMPLATE = """
You are the Research Coordinator Agent for FATLAS.

Your job:
Combine the findings from all specialist agents and identify the most meaningful shared mechanisms,
conflicts, gaps, and promising safe research directions.

Project topic:
{research_topic}

Project goal:
{project_goal}

Specialist reports:
{specialist_reports_json}

Instructions:
- Build an integrated systems model.
- Identify overlapping pathways across hormone, genetics, microbiome, inflammation, and aging domains.
- Look for convergence, not just isolated findings.
- Separate strong evidence from hypothesis.
- Do not make medical treatment claims.
- Do not recommend unsafe weight-loss methods.
- Prioritize safety and testability.
- Produce a scientist-friendly research synthesis.

Return ONLY valid JSON in this schema:

{{
  "project_title": "{project_title}",
  "coordinator_summary": "",
  "integrated_model": {{
    "plain_language_model": "",
    "mechanistic_model": "",
    "central_hypothesis": ""
  }},
  "most_relevant_converging_pathways": [
    {{
      "pathway": "",
      "domains_connected": [],
      "why_it_matters": "",
      "evidence_strength": "high | moderate | emerging | speculative"
    }}
  ],
  "candidate_switches_or_control_points": [
    {{
      "control_point": "",
      "why_it_may_reduce_visceral_fat_signaling": "",
      "safety_concerns": "",
      "research_priority": "high | medium | low"
    }}
  ],
  "knowledge_gaps": [],
  "recommended_experiments": [
    {{
      "experiment": "",
      "purpose": "",
      "data_needed": "",
      "success_signal": "",
      "ethical_or_safety_considerations": ""
    }}
  ],
  "biomarker_panel_candidates": [],
  "safe_near_term_public_health_angles": [],
  "unsafe_or_overhyped_angles_to_avoid": [],
  "next_agent_tasks": [
    {{
      "agent": "",
      "task": "",
      "reason": ""
    }}
  ]
}}
"""

In [ ]:
def run_research_coordinator(reports):
    reports_json = json.dumps(reports, indent=2)

    prompt = COORDINATOR_PROMPT_TEMPLATE.format(
        project_title=PROJECT_TITLE,
        research_topic=RESEARCH_TOPIC,
        project_goal=PROJECT_GOAL,
        specialist_reports_json=reports_json
    )

    raw = call_gemini(prompt, temperature=0.15)
    try:
        data = extract_json(raw)
    except Exception as e:
        print("Coordinator JSON parsing failed. Saving raw output.")
        data = {
            "project_title": PROJECT_TITLE,
            "parse_error": str(e),
            "raw_output": raw
        }

    save_json(data, "research_coordinator_synthesis.json")
    return data


coordinator_report = run_research_coordinator(specialist_reports)
coordinator_report

## 10. Generate a readable research briefing

This converts the Coordinator report into a Markdown briefing.

In [ ]:
def make_research_briefing(coordinator_report, specialist_reports):
    lines = []
    lines.append(f"# {coordinator_report.get('project_title', PROJECT_TITLE)}")
    lines.append("")
    lines.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    lines.append("")
    lines.append("## Coordinator Summary")
    lines.append("")
    lines.append(coordinator_report.get("coordinator_summary", ""))
    lines.append("")

    model = coordinator_report.get("integrated_model", {})
    lines.append("## Integrated Model")
    lines.append("")
    lines.append("### Plain-language model")
    lines.append(model.get("plain_language_model", ""))
    lines.append("")
    lines.append("### Mechanistic model")
    lines.append(model.get("mechanistic_model", ""))
    lines.append("")
    lines.append("### Central hypothesis")
    lines.append(model.get("central_hypothesis", ""))
    lines.append("")

    lines.append("## Most Relevant Converging Pathways")
    lines.append("")
    for item in coordinator_report.get("most_relevant_converging_pathways", []):
        lines.append(f"### {item.get('pathway', '')}")
        lines.append(f"- Domains connected: {', '.join(item.get('domains_connected', []))}")
        lines.append(f"- Why it matters: {item.get('why_it_matters', '')}")
        lines.append(f"- Evidence strength: {item.get('evidence_strength', '')}")
        lines.append("")

    lines.append("## Candidate Switches or Control Points")
    lines.append("")
    for item in coordinator_report.get("candidate_switches_or_control_points", []):
        lines.append(f"### {item.get('control_point', '')}")
        lines.append(f"- Rationale: {item.get('why_it_may_reduce_visceral_fat_signaling', '')}")
        lines.append(f"- Safety concerns: {item.get('safety_concerns', '')}")
        lines.append(f"- Research priority: {item.get('research_priority', '')}")
        lines.append("")

    lines.append("## Knowledge Gaps")
    lines.append("")
    for gap in coordinator_report.get("knowledge_gaps", []):
        lines.append(f"- {gap}")
    lines.append("")

    lines.append("## Recommended Experiments")
    lines.append("")
    for item in coordinator_report.get("recommended_experiments", []):
        lines.append(f"### {item.get('experiment', '')}")
        lines.append(f"- Purpose: {item.get('purpose', '')}")
        lines.append(f"- Data needed: {item.get('data_needed', '')}")
        lines.append(f"- Success signal: {item.get('success_signal', '')}")
        lines.append(f"- Ethics/safety: {item.get('ethical_or_safety_considerations', '')}")
        lines.append("")

    lines.append("## Candidate Biomarker Panel")
    lines.append("")
    for biomarker in coordinator_report.get("biomarker_panel_candidates", []):
        lines.append(f"- {biomarker}")
    lines.append("")

    lines.append("## Safe Near-Term Public Health Angles")
    lines.append("")
    for angle in coordinator_report.get("safe_near_term_public_health_angles", []):
        lines.append(f"- {angle}")
    lines.append("")

    lines.append("## Unsafe or Overhyped Angles to Avoid")
    lines.append("")
    for angle in coordinator_report.get("unsafe_or_overhyped_angles_to_avoid", []):
        lines.append(f"- {angle}")
    lines.append("")

    lines.append("## Next Agent Tasks")
    lines.append("")
    for task in coordinator_report.get("next_agent_tasks", []):
        lines.append(f"- **{task.get('agent', '')}:** {task.get('task', '')} Reason: {task.get('reason', '')}")
    lines.append("")

    lines.append("## Specialist Reports Included")
    lines.append("")
    for report in specialist_reports:
        lines.append(f"- {report.get('agent_name', '')}")

    return "\\n".join(lines)


briefing_md = make_research_briefing(coordinator_report, specialist_reports)
briefing_path = save_markdown(briefing_md, "FATLAS_research_briefing.md")

print(f"Saved briefing to: {briefing_path}")
print("\\nPreview:\\n")
print(briefing_md[:2000])

## 10B. Generate a clean HTML intelligence briefing

This creates an easy-to-read card-style HTML report instead of a long Markdown document.  
It is better for iPad reading and easier to share.

In [ ]:
from html import escape

def list_items(items):
    if not items:
        return "<p class='muted'>No items returned.</p>"
    html = "<ul>"
    for item in items:
        html += f"<li>{escape(str(item))}</li>"
    html += "</ul>"
    return html

def card(title, body, badge=None):
    badge_html = f"<span class='badge'>{escape(badge)}</span>" if badge else ""
    return f"""
    <section class="card">
      <h2>{escape(title)} {badge_html}</h2>
      <div>{body}</div>
    </section>
    """

def make_html_briefing(coordinator_report, specialist_reports):
    model = coordinator_report.get("integrated_model", {})
    pathways = coordinator_report.get("most_relevant_converging_pathways", [])
    switches = coordinator_report.get("candidate_switches_or_control_points", [])
    experiments = coordinator_report.get("recommended_experiments", [])
    tasks = coordinator_report.get("next_agent_tasks", [])

    pathway_cards = ""
    for item in pathways:
        pathway_cards += card(
            item.get("pathway", "Unnamed pathway"),
            f"""
            <p><strong>Domains connected:</strong> {escape(", ".join(item.get("domains_connected", [])))}</p>
            <p><strong>Why it matters:</strong> {escape(item.get("why_it_matters", ""))}</p>
            """,
            item.get("evidence_strength", "")
        )

    switch_cards = ""
    for item in switches:
        switch_cards += card(
            item.get("control_point", "Unnamed control point"),
            f"""
            <p><strong>Why it may matter:</strong> {escape(item.get("why_it_may_reduce_visceral_fat_signaling", ""))}</p>
            <p><strong>Safety concerns:</strong> {escape(item.get("safety_concerns", ""))}</p>
            """,
            f"Priority: {item.get('research_priority', '')}"
        )

    experiment_cards = ""
    for item in experiments:
        experiment_cards += card(
            item.get("experiment", "Unnamed experiment"),
            f"""
            <p><strong>Purpose:</strong> {escape(item.get("purpose", ""))}</p>
            <p><strong>Data needed:</strong> {escape(item.get("data_needed", ""))}</p>
            <p><strong>Success signal:</strong> {escape(item.get("success_signal", ""))}</p>
            <p><strong>Ethics/Safety:</strong> {escape(item.get("ethical_or_safety_considerations", ""))}</p>
            """
        )

    task_cards = ""
    for item in tasks:
        task_cards += card(
            item.get("agent", "Agent"),
            f"""
            <p><strong>Task:</strong> {escape(item.get("task", ""))}</p>
            <p><strong>Reason:</strong> {escape(item.get("reason", ""))}</p>
            """
        )

    specialist_rows = ""
    for report in specialist_reports:
        specialist_rows += f"""
        <tr>
          <td>{escape(report.get("agent_name", ""))}</td>
          <td>{escape(report.get("executive_summary", ""))}</td>
        </tr>
        """

    html = f"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<title>{escape(PROJECT_TITLE)}</title>
<style>
  body {{
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Arial, sans-serif;
    background: #f4f6f8;
    color: #1f2937;
    margin: 0;
    padding: 0;
    line-height: 1.55;
  }}
  header {{
    background: linear-gradient(135deg, #172554, #2563eb);
    color: white;
    padding: 36px 28px;
  }}
  header h1 {{
    margin: 0 0 8px 0;
    font-size: 34px;
    line-height: 1.1;
  }}
  header p {{
    margin: 0;
    font-size: 17px;
    opacity: 0.95;
  }}
  main {{
    max-width: 1100px;
    margin: 0 auto;
    padding: 24px;
  }}
  .grid {{
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(280px, 1fr));
    gap: 18px;
  }}
  .card {{
    background: white;
    border-radius: 18px;
    padding: 22px;
    margin-bottom: 18px;
    box-shadow: 0 8px 22px rgba(15, 23, 42, 0.08);
    border: 1px solid #e5e7eb;
  }}
  .card h2 {{
    margin-top: 0;
    font-size: 22px;
    line-height: 1.2;
  }}
  .badge {{
    display: inline-block;
    background: #e0f2fe;
    color: #075985;
    border-radius: 999px;
    padding: 4px 10px;
    font-size: 13px;
    margin-left: 8px;
    vertical-align: middle;
  }}
  .section-title {{
    font-size: 28px;
    margin: 32px 0 14px 0;
    color: #111827;
  }}
  .hero {{
    background: white;
    border-radius: 22px;
    padding: 26px;
    margin-top: -10px;
    box-shadow: 0 10px 28px rgba(15, 23, 42, 0.12);
  }}
  .muted {{
    color: #6b7280;
  }}
  table {{
    width: 100%;
    border-collapse: collapse;
    background: white;
    border-radius: 16px;
    overflow: hidden;
    box-shadow: 0 8px 22px rgba(15, 23, 42, 0.08);
  }}
  th, td {{
    padding: 14px;
    border-bottom: 1px solid #e5e7eb;
    vertical-align: top;
    text-align: left;
  }}
  th {{
    background: #eff6ff;
  }}
  footer {{
    text-align: center;
    color: #6b7280;
    padding: 30px;
  }}
</style>
</head>
<body>
<header>
  <h1>FATLAS Intelligence Briefing</h1>
  <p>{escape(PROJECT_TITLE)} · Generated {datetime.now().strftime('%Y-%m-%d %H:%M')}</p>
</header>

<main>
  <section class="hero">
    <h2>Executive Summary</h2>
    <p>{escape(coordinator_report.get("coordinator_summary", ""))}</p>
  </section>

  <h2 class="section-title">Integrated Model</h2>
  <div class="grid">
    {card("Plain-English Model", f"<p>{escape(model.get('plain_language_model', ''))}</p>")}
    {card("Mechanistic Model", f"<p>{escape(model.get('mechanistic_model', ''))}</p>")}
    {card("Central Hypothesis", f"<p>{escape(model.get('central_hypothesis', ''))}</p>")}
  </div>

  <h2 class="section-title">Top Converging Pathways</h2>
  <div class="grid">
    {pathway_cards}
  </div>

  <h2 class="section-title">Candidate Switches or Control Points</h2>
  <div class="grid">
    {switch_cards}
  </div>

  <h2 class="section-title">Knowledge Gaps</h2>
  {card("What We Still Do Not Know", list_items(coordinator_report.get("knowledge_gaps", [])))}

  <h2 class="section-title">Recommended Experiments</h2>
  <div class="grid">
    {experiment_cards}
  </div>

  <h2 class="section-title">Candidate Biomarker Panel</h2>
  {card("Biomarkers to Track", list_items(coordinator_report.get("biomarker_panel_candidates", [])))}

  <h2 class="section-title">Safe Near-Term Public Health Angles</h2>
  {card("Practical, Lower-Risk Angles", list_items(coordinator_report.get("safe_near_term_public_health_angles", [])))}

  <h2 class="section-title">Unsafe or Overhyped Angles to Avoid</h2>
  {card("Do Not Overclaim These", list_items(coordinator_report.get("unsafe_or_overhyped_angles_to_avoid", [])))}

  <h2 class="section-title">Next Agent Tasks</h2>
  <div class="grid">
    {task_cards}
  </div>

  <h2 class="section-title">Specialist Agent Summaries</h2>
  <table>
    <thead>
      <tr>
        <th>Agent</th>
        <th>Summary</th>
      </tr>
    </thead>
    <tbody>
      {specialist_rows}
    </tbody>
  </table>
</main>

<footer>
  FATLAS research output for hypothesis generation only. Not medical advice.
</footer>
</body>
</html>
"""
    return html

html_report = make_html_briefing(coordinator_report, specialist_reports)
html_path = OUTPUT_DIR / "FATLAS_intelligence_briefing.html"
html_path.write_text(html_report, encoding="utf-8")

print(f"Saved HTML briefing to: {html_path}")

if IN_COLAB:
    from IPython.display import HTML, display
    display(HTML(html_report))

## 11. Optional: create a systems-map prompt

You can paste this into an image generator or diagram tool to create a visual map.

In [ ]:
central_hypothesis = coordinator_report.get("integrated_model", {}).get("central_hypothesis", "")

SYSTEMS_MAP_PROMPT = f"""
Create a clean scientific systems diagram titled:
{PROJECT_TITLE}

Show how these domains converge on visceral abdominal fat storage:
1. Hormone signaling
2. Genetics and epigenetics
3. Gut microbiome
4. Inflammation and immune activity
5. Aging and metabolic change

Use the Research Coordinator central hypothesis:
{central_hypothesis}

Show arrows between:
stress, diet, sleep, genetics, aging, gut barrier, microbiome metabolites,
insulin resistance, cortisol signaling, leptin resistance, adipose tissue inflammation,
mitochondrial dysfunction, adipocyte hypertrophy, macrophage infiltration,
visceral fat expansion, fatty liver risk, diabetes risk, cardiovascular risk.

Make it readable, medically serious, and not sensationalized.
"""

print(SYSTEMS_MAP_PROMPT)
save_markdown(SYSTEMS_MAP_PROMPT, "systems_map_prompt.txt")

## 12. Zip the output files for download

After running the notebook, this creates a ZIP file with the reports.

In [ ]:
import shutil

zip_path = shutil.make_archive("fatlas_outputs", "zip", OUTPUT_DIR)
print("Created:", zip_path)

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 13. Suggested next upgrade

After this basic version works, the next version can add:

- PubMed search
- Paper ingestion
- Citation verification
- A knowledge graph
- Human review scoring
- Export to GitHub Pages
- Versioned `findings.json`
- A dashboard showing pathways, confidence, and open questions

This first notebook is intentionally simple so you can run it quickly and understand every piece.